# Clinic-Noise Playground — FLEURS → hospital-environment degradation

A **modular testbed** for designing the realistic clinic/hospital noise we apply to FLEURS. Run on Kaggle (CPU is enough — no GPU), download `/kaggle/working/fleurs_replicated_medical_noise/`, and listen.

### How to use it
1. Attach noise banks via **Add Data**: **MUSAN** (babble), **ESC-50** (cough/breath). **DEMAND** optional (ambient/HVAC). Missing banks are skipped gracefully — effects that need them just no-op.
2. Get the 10 base clips on Kaggle: upload your `fleurs_clean/clip_00–09.wav` as a dataset (auto-detected), or leave `BASE_DIR=""` to pull the first 10 from FLEURS.
3. **Edit the `PIPELINE` in the CONFIG cell** — that's the whole experiment surface.
4. Run All → download the output folder → listen.

### The design (why it's scalable)
- Every degradation is one **effect function** `fn(audio, sr, rng, **params)`, registered in `EFFECTS` with a category (`reverb` / `noise` / `mic`).
- The **`PIPELINE`** is just a list of `(effect_name, params)`. Reorder, comment out, duplicate, or retune any line freely.
- **Add a brand-new noise** in 3 steps: write a function in the ENGINE cell → add it to `EFFECTS` with a category → add a line to `PIPELINE`.
- Outputs include **isolated stems** (`reverb` / `noise` / `mic`) plus the `full` mix and the `clean` reference, so you can judge each effect on its own.

> Signal chain (physical order): **clean speech → room reverb → + real ambient noise → mic/codec degradation.**

Install the two non-default libs (ffmpeg + scipy are already on Kaggle).

In [ ]:
!pip install -q soundfile pyroomacoustics

Imports.

In [ ]:
import os, io, csv, glob, subprocess, tempfile, warnings
from math import gcd
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly, butter, sosfilt
warnings.filterwarnings("ignore")
print("imports ok")

## ✏️ CONFIG — this is the only cell you normally edit
The `PIPELINE` *is* the experiment. Each line is `(effect_name, params)`.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  EXPERIMENT CONFIG
# ════════════════════════════════════════════════════════════════
SR        = 16000
N_CLIPS   = 10
SEED      = 7
BASE_DIR  = ""        # "" = auto-detect uploaded clip_*.wav, else FLEURS first N
OUT_DIR   = "/kaggle/working/fleurs_replicated_medical_noise"
STEM_MODE = "grouped" # "grouped" (clean+reverb+noise+mic+full) | "per_effect" | "full_only"

# ── THE PIPELINE — reorder / comment-out / duplicate / retune freely ──
# To add a new effect: define it in the ENGINE cell, register it in EFFECTS
# with a category, then add a line here.
PIPELINE = [
    ("reverb_pyroom",   dict(rt60=0.45, room=(4.0, 5.0, 3.0))),     # small tiled exam room
    ("add_babble",      dict(snr_db=15, n_voices=4)),               # waiting-room chatter
    ("add_events",      dict(bank="cough", snr_db=8, n_events=2)),  # patient cough/breath
    ("synth_hum",       dict(snr_db=28, base=50.0, n_harm=4)),      # HVAC / mains hum
    ("synth_beeps",     dict(snr_db=22, freq=1000.0, interval=5.0)),# monitor beep
    ("bandlimit",       dict(low=120.0, high=6000.0)),              # cheap-mic rolloff
    ("clip_dist",       dict(drive=0.15)),                          # mild overdrive
    ("codec_roundtrip", dict(codec="opus", bitrate="20k")),         # VoIP / cheap capture
]
# ════════════════════════════════════════════════════════════════
print(f"{len(PIPELINE)} pipeline steps | stems={STEM_MODE} | out={OUT_DIR}")

---
## ⚙️ ENGINE — you rarely need to touch the cells below

In [ ]:
# ── audio IO + level helpers ──────────────────────────────────────
def load_wav(path, sr_target):
    x, sr = sf.read(path, dtype="float32")
    if x.ndim > 1:
        x = x.mean(axis=1).astype(np.float32)
    if sr != sr_target and len(x):
        g = gcd(int(sr), int(sr_target))
        x = resample_poly(x, sr_target // g, sr // g).astype(np.float32)
    return x, sr_target

def save_wav(path, x, sr):
    sf.write(path, np.clip(x, -1.0, 1.0).astype(np.float32), sr, subtype="PCM_16")

def _rms(x):
    return float(np.sqrt(np.mean(np.asarray(x, np.float64) ** 2)) + 1e-12)

def scale_to_snr(speech, noise, snr_db):
    """Scale `noise` so SNR(speech, noise) == snr_db (dB)."""
    target = _rms(speech) / (10.0 ** (snr_db / 20.0))
    return (noise * (target / _rms(noise))).astype(np.float32)

def _fit(x, n):
    """Loop/trim x to exactly n samples."""
    if len(x) == 0:
        return np.zeros(n, np.float32)
    if len(x) < n:
        x = np.tile(x, int(np.ceil(n / len(x))))
    return x[:n].astype(np.float32)
print("io helpers ok")

In [ ]:
# ── noise banks + auto-discovery from /kaggle/input ───────────────
class NoiseBank:
    def __init__(self, files, sr):
        self.files = list(files); self.sr = sr
    def _read(self, path):
        try:
            x, _ = load_wav(path, self.sr); return x
        except Exception:
            return np.zeros(0, np.float32)
    def sample(self, n, rng):
        """Continuous noise of length n (random file, random offset, looped)."""
        if not self.files:
            return np.zeros(n, np.float32)
        x = self._read(str(rng.choice(self.files)))
        if len(x) == 0:
            return np.zeros(n, np.float32)
        if len(x) > n:
            s = int(rng.integers(0, len(x) - n + 1)); x = x[s:s + n]
        return _fit(x, n)
    def event(self, rng):
        """A whole short clip (for discrete events like a cough)."""
        if not self.files:
            return np.zeros(0, np.float32)
        return self._read(str(rng.choice(self.files)))

def _walk_wavs(root, exts=(".wav", ".flac", ".mp3", ".ogg")):
    if not root or not os.path.isdir(root):
        return []
    out = []
    for dp, _, fns in os.walk(root):
        for f in fns:
            if f.lower().endswith(exts):
                out.append(os.path.join(dp, f))
    return out

def find_input_dir(*keys):
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return None
    for d in sorted(os.listdir(base)):
        if any(k in d.lower() for k in keys):
            return os.path.join(base, d)
    return None

def _subdir(root, name):
    if not root:
        return None
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d.lower() == name:
                return os.path.join(dp, d)
    return root

def esc50_bank(root, categories, sr):
    """ESC-50: filter by category via its meta CSV; fall back to all wavs."""
    cat = {}
    for dp, _, fns in os.walk(root or ""):
        for f in fns:
            if f.lower().endswith(".csv"):
                try:
                    for r in csv.DictReader(open(os.path.join(dp, f), encoding="utf-8")):
                        if "filename" in r and "category" in r:
                            cat[r["filename"]] = r["category"]
                except Exception:
                    pass
    allw = _walk_wavs(root)
    if cat and categories:
        want = set(categories)
        files = [w for w in allw if cat.get(os.path.basename(w)) in want]
    else:
        files = allw
    return NoiseBank(files, sr)

# ── recursive discovery: handles datasets nested under /kaggle/input/<bundle>/<owner>/... ──
def find_dir(*keys, base="/kaggle/input"):
    for dp, dns, _ in os.walk(base):
        for d in dns:
            if any(k in d.lower() for k in keys):
                return os.path.join(dp, d)
    return None

_musan  = find_dir("musan")
_esc    = find_dir("esc50", "esc-50", "environmental-sound")
_demand = find_dir("demand")

BANKS = {}
BANKS["babble"]  = NoiseBank(_walk_wavs(_subdir(_musan, "speech")), SR)
_amb = _walk_wavs(_subdir(_musan, "noise")) + _walk_wavs(_demand)
BANKS["ambient"] = NoiseBank(_amb, SR)
BANKS["cough"]   = esc50_bank(_esc, ["coughing", "breathing", "sneezing"], SR)

print("Banks discovered:")
print(f"  musan={_musan}  esc50={_esc}  demand={_demand}")
for k, v in BANKS.items():
    print(f"  bank {k:8}: {len(v.files)} files")

In [ ]:
import os, collections

BASE = "/kaggle/input"
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg")

print("=" * 70)
print("ATTACHED DATASETS (top level of /kaggle/input)")
print("=" * 70)
if not os.path.isdir(BASE):
    print("  !! /kaggle/input does not exist — nothing attached")
else:
    tops = sorted(os.listdir(BASE))
    if not tops:
        print("  !! /kaggle/input is empty — no datasets attached")
    for d in tops:
        print(f"  • {d}")

print()
print("=" * 70)
print("PER-DATASET BREAKDOWN")
print("=" * 70)
for d in sorted(os.listdir(BASE)):
    root = os.path.join(BASE, d)
    if not os.path.isdir(root):
        continue
    n_audio = 0
    subdirs = set()
    ext_counts = collections.Counter()
    sample_files = []
    for dp, dns, fns in os.walk(root):
        rel = os.path.relpath(dp, root)
        if rel != ".":
            subdirs.add(rel.split(os.sep)[0])   # first-level subdir name
        for f in fns:
            ext = os.path.splitext(f)[1].lower()
            ext_counts[ext] += 1
            if ext in AUDIO_EXTS:
                n_audio += 1
                if len(sample_files) < 3:
                    sample_files.append(os.path.relpath(os.path.join(dp, f), root))
    print(f"\n### {d}")
    print(f"    path        : {root}")
    print(f"    audio files : {n_audio}")
    print(f"    1st-level subdirs: {sorted(subdirs)[:15]}")
    print(f"    file types  : {dict(ext_counts.most_common(8))}")
    if sample_files:
        print(f"    sample audio: {sample_files}")

In [ ]:
# ── EFFECT FUNCTIONS: fn(audio, sr, rng, **params) -> audio ───────
# REVERB ----------------------------------------------------------------
def reverb_synth(audio, sr, rng, rt60=0.4, **k):
    n = max(1, int(sr * rt60))
    t = np.arange(n)
    ir = (rng.standard_normal(n) * np.exp(-6.908 * t / (rt60 * sr))).astype(np.float32)
    ir[0] += 1.0                                   # direct path
    out = np.convolve(audio, ir)[:len(audio)]
    return (out * (_rms(audio) / _rms(out))).astype(np.float32)

def reverb_pyroom(audio, sr, rng, rt60=0.4, room=(4.0, 5.0, 3.0), **k):
    try:
        import pyroomacoustics as pra
        e_abs, max_order = pra.inverse_sabine(rt60, list(room))
        r = pra.ShoeBox(list(room), fs=sr, materials=pra.Material(e_abs),
                        max_order=int(max_order))
        src = [room[0] * 0.5, room[1] * 0.35, 1.2]
        mic = [room[0] * 0.5, room[1] * 0.65, 1.2]
        r.add_source(src, signal=audio.astype(np.float64))
        r.add_microphone(np.array(mic).reshape(3, 1))
        r.simulate()
        out = np.asarray(r.mic_array.signals[0], np.float32)[:len(audio)]
        out = _fit(out, len(audio))
        return (out * (_rms(audio) / _rms(out))).astype(np.float32)
    except Exception as e:
        print(f"    [reverb_pyroom -> synth fallback: {e}]")
        return reverb_synth(audio, sr, rng, rt60=rt60)

# NOISE -----------------------------------------------------------------
def add_noise(audio, sr, rng, bank="ambient", snr_db=15, **k):
    nb = BANKS.get(bank)
    if not nb or not nb.files:
        return audio
    return (audio + scale_to_snr(audio, nb.sample(len(audio), rng), snr_db)).astype(np.float32)

def add_babble(audio, sr, rng, snr_db=15, n_voices=4, bank="babble", **k):
    nb = BANKS.get(bank)
    if not nb or not nb.files:
        return audio
    mix = np.zeros(len(audio), np.float32)
    for _ in range(n_voices):
        mix += nb.sample(len(audio), rng)
    return (audio + scale_to_snr(audio, mix, snr_db)).astype(np.float32)

def add_events(audio, sr, rng, bank="cough", snr_db=10, n_events=2, **k):
    nb = BANKS.get(bank)
    if not nb or not nb.files:
        return audio
    out = audio.copy()
    for _ in range(n_events):
        ev = nb.event(rng)
        if len(ev) == 0:
            continue
        ev = ev[:len(audio)]
        pos = int(rng.integers(0, max(1, len(audio) - len(ev))))
        seg = out[pos:pos + len(ev)]
        ev = scale_to_snr(audio, ev[:len(seg)], snr_db)
        out[pos:pos + len(ev)] += ev
    return out.astype(np.float32)

def synth_hum(audio, sr, rng, snr_db=28, base=50.0, n_harm=4, **k):
    t = np.arange(len(audio)) / sr
    hum = sum(np.sin(2 * np.pi * base * h * t) / h for h in range(1, n_harm + 1))
    return (audio + scale_to_snr(audio, hum.astype(np.float32), snr_db)).astype(np.float32)

def synth_beeps(audio, sr, rng, snr_db=22, freq=1000.0, beep_ms=150, interval=5.0, **k):
    bn = int(sr * beep_ms / 1000.0)
    beep = (np.sin(2 * np.pi * freq * np.arange(bn) / sr) * np.hanning(bn)).astype(np.float32)
    tmpl = np.zeros(len(audio), np.float32); step = max(bn, int(sr * interval))
    for pos in range(0, len(audio) - bn, step):
        tmpl[pos:pos + bn] += beep
    return (audio + scale_to_snr(audio, tmpl, snr_db)).astype(np.float32)

# MIC / CHANNEL ---------------------------------------------------------
def bandlimit(audio, sr, rng, low=120.0, high=6000.0, order=4, **k):
    high = min(high, sr / 2 - 1)
    sos = butter(order, [low, high], btype="band", fs=sr, output="sos")
    return sosfilt(sos, audio).astype(np.float32)

def clip_dist(audio, sr, rng, drive=0.2, **k):
    return np.clip(audio * (1.0 + drive * 6.0), -1.0, 1.0).astype(np.float32)

def bitdepth(audio, sr, rng, bits=8, **k):
    q = 2 ** (bits - 1)
    return (np.round(np.clip(audio, -1, 1) * q) / q).astype(np.float32)

_CODECS = {
    "opus": ("opus", ["-c:a", "libopus"]),
    "mp3":  ("mp3",  ["-c:a", "libmp3lame"]),
    "aac":  ("m4a",  ["-c:a", "aac"]),
    "amr":  ("amr",  ["-c:a", "libopencore_amrnb", "-ar", "8000", "-ac", "1"]),
}
def codec_roundtrip(audio, sr, rng, codec="opus", bitrate="24k", **k):
    ext, enc = _CODECS.get(codec, _CODECS["opus"])
    try:
        with tempfile.TemporaryDirectory() as d:
            wi, ec, wo = (os.path.join(d, f) for f in ("in.wav", "e." + ext, "out.wav"))
            save_wav(wi, audio, sr)
            subprocess.run(["ffmpeg", "-y", "-i", wi, *enc, "-b:a", bitrate, ec],
                           check=True, capture_output=True)
            subprocess.run(["ffmpeg", "-y", "-i", ec, "-ar", str(sr), "-ac", "1", wo],
                           check=True, capture_output=True)
            y, _ = load_wav(wo, sr)
        return _fit(y, len(audio))
    except Exception as e:
        print(f"    [codec_roundtrip skipped: {e}]")
        return audio
print("effects defined")

In [ ]:
# ── REGISTRY (name -> (fn, category)) and pipeline runner ─────────
EFFECTS = {
    "reverb_pyroom":   (reverb_pyroom,   "reverb"),
    "reverb_synth":    (reverb_synth,    "reverb"),
    "add_noise":       (add_noise,       "noise"),
    "add_babble":      (add_babble,      "noise"),
    "add_events":      (add_events,      "noise"),
    "synth_hum":       (synth_hum,       "noise"),
    "synth_beeps":     (synth_beeps,     "noise"),
    "bandlimit":       (bandlimit,       "mic"),
    "clip_dist":       (clip_dist,       "mic"),
    "bitdepth":        (bitdepth,        "mic"),
    "codec_roundtrip": (codec_roundtrip, "mic"),
}

def run_steps(audio, steps, seed):
    rng = np.random.default_rng(seed)
    x = audio.copy()
    for name, params in steps:
        fn, _ = EFFECTS[name]
        try:
            x = fn(x, SR, rng, **params)
        except Exception as e:
            print(f"    [warn] {name} failed: {e}")
    return x

def pipe_str(steps):
    return " -> ".join(n for n, _ in steps)
print("registry ok |", len(EFFECTS), "effects available:", ", ".join(EFFECTS))

In [ ]:
# ── load the 10 base clips (uploaded dir, else FLEURS first N) ────
def load_base_clips():
    src = BASE_DIR
    if not src:
        for dp, _, fns in os.walk("/kaggle/input"):
            cs = sorted(f for f in fns if f.startswith("clip_") and f.endswith(".wav"))
            if cs and ("fleurs" in dp.lower() or len(cs) >= 5):
                src = dp; break
    if src and os.path.isdir(src):
        print("base clips from:", src)
        out = []
        for f in sorted(os.listdir(src)):
            if f.endswith(".wav"):
                x, _ = load_wav(os.path.join(src, f), SR)
                out.append((f, x))
        return out[:N_CLIPS]
    print(f"no uploaded clips found - pulling first {N_CLIPS} FLEURS fa_ir test clips")
    from datasets import load_dataset, Audio
    ds = load_dataset("google/fleurs", "fa_ir", split="test").cast_column("audio", Audio(decode=False))
    out = []
    for i, row in enumerate(ds):
        if i >= N_CLIPS:
            break
        a = row["audio"]
        if a.get("bytes"):
            y, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
        else:
            y, sr = sf.read(a["path"], dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != SR:
            g = gcd(int(sr), SR); y = resample_poly(y, SR // g, sr // g)
        out.append((f"clip_{i:02d}.wav", y.astype(np.float32)))
    return out

base_clips = load_base_clips()
print(f"loaded {len(base_clips)} base clips")

In [ ]:
# ── RUN: apply pipeline + write stems + manifest ─────────────────
os.makedirs(OUT_DIR, exist_ok=True)
for f in glob.glob(os.path.join(OUT_DIR, "*")):
    os.remove(f)

manifest = []
for i, (name, audio) in enumerate(base_clips):
    base = os.path.splitext(name)[0]
    seed = SEED + i

    save_wav(os.path.join(OUT_DIR, f"{base}_clean.wav"), audio, SR)
    manifest.append((base, "clean", ""))

    full = run_steps(audio, PIPELINE, seed)
    save_wav(os.path.join(OUT_DIR, f"{base}_full.wav"), full, SR)
    manifest.append((base, "full", pipe_str(PIPELINE)))

    if STEM_MODE == "grouped":
        for cat in ("reverb", "noise", "mic"):
            sub = [(n, p) for (n, p) in PIPELINE if EFFECTS[n][1] == cat]
            if sub:
                y = run_steps(audio, sub, seed)
                save_wav(os.path.join(OUT_DIR, f"{base}_{cat}.wav"), y, SR)
                manifest.append((base, cat, pipe_str(sub)))
    elif STEM_MODE == "per_effect":
        for j, (n, p) in enumerate(PIPELINE):
            y = run_steps(audio, [(n, p)], seed)
            save_wav(os.path.join(OUT_DIR, f"{base}_{j:02d}_{n}.wav"), y, SR)
            manifest.append((base, f"{j:02d}_{n}", str(p)))

    print(f"[{i+1}/{len(base_clips)}] {base}")

with open(os.path.join(OUT_DIR, "manifest.csv"), "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["clip", "stem", "steps"]); w.writerows(manifest)

print(f"\nDone. {len(manifest)} files -> {OUT_DIR}")
print("Download that folder and listen. Pipeline:")
print("  " + pipe_str(PIPELINE))

## Iterate
Tune the `PIPELINE` in the CONFIG cell, re-run, re-download, re-listen. Quick knobs:
- **More/less of a noise:** lower `snr_db` = louder noise.
- **Bigger room / more echo:** raise `rt60`.
- **Worse mic:** lower codec `bitrate`, narrow `bandlimit`, raise `clip_dist` `drive`.
- **Disable an effect:** comment out its line. **A/B a noise alone:** keep `STEM_MODE="grouped"` and listen to the `*_noise.wav` / `*_reverb.wav` / `*_mic.wav` stems.
- **New noise type:** add a function in the effects cell → register in `EFFECTS` with a category → add a `PIPELINE` line.